|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 2. You can batch the requests, and you can schedule them
once for each iteration. Now the failures come from many users at the same
time: padding, masks, queues and batch sizes.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 05. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 2.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| GPU | Memory | Bandwidth | bf16 compute |
|---|---|---|---|
| A100 SXM 80GB | 80 GB | 2,039 GB/s | 312 TFLOP/s |
| L40S | 48 GB | 864 GB/s | 362 TFLOP/s |

| Model | Layers | KV heads | head_dim | bf16 weights | KV bytes per token |
|---|---|---|---|---|---|
| Qwen3-1.7B | 28 | 8 | 128 | 3.44 GB | 114,688 (112 KiB) |
| Llama-3-8B | 32 | 8 | 128 | 16.1 GB | 131,072 (128 KiB) |

Three formulas from Parts 1 and 2:

    decode step (s)  >= (weight bytes + B x context x KV bytes per token) / bandwidth
    decode FLOP      =  2 x parameters x B
    Little's law     :  requests in the system = arrival rate x time in the system

# Ticket 1: the short prompts get garbage

**Severity:** high. **Reported by:** the team that moved to batching.

> We batch 8 prompts now. The longest prompt of each batch gets a good
> answer. The other prompts get garbage. When we run them one at a time,
> all of them are good.

**Evidence**

- The batching code:

  ```python
  batch = tokenizer(prompts, padding=True, return_tensors='pt')
  out = model(batch.input_ids, attention_mask=batch.attention_mask, use_cache=True)
  next_tokens = out.logits[:, -1].argmax(-1)
  ```

- `print(tokenizer.padding_side)` prints `right`.
- A sample batch: the prompt lengths are 31, 24, 31, 12, 18, 31, 27, 9.
  The three prompts of length 31 get good answers.
- The GPU ran out of memory one time last week, at batch 16.

### Solution

- **Root cause.** The tokenizer pads on the right. Then position -1 of a
  short row is a pad token, and `logits[:, -1]` reads the prediction
  after a pad, not after the last real token.
- **The number.** The bad rows are exactly the rows with padding: 5 rows
  with 22, 19, 13, 4 and 7 pads. The rows with 0 pads are good. A row of
  length 31 is correct because its position -1 is real.
- **The fix.** Set `tokenizer.padding_side = 'left'`, as your stage 04
  does. Then position -1 is the last real token of every row. The other
  fix: read the logits at index `length - 1` of each row.
- **The guard.** A test with a batch of mixed lengths: each row must
  match the same prompt run alone (in fp32).

**The noise.** The OOM at batch 16. It is a capacity limit, and it has
nothing to do with the order of the tokens.

In [ ]:
lengths = [31, 24, 31, 12, 18, 31, 27, 9]
longest = max(lengths)
for length in lengths:
    pads = longest - length
    print(f'length {length:2d}: {pads:2d} pads, position -1 is', 'a real token' if pads == 0 else 'a PAD')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What is the last token of each row of `batch.input_ids`?*
  For the three rows of length 31, a real token. For the other five rows,
  151643, the pad token.
- *Are the answers of the short prompts wrong from the first token?*
  Yes. The first token is already wrong for those five rows.
- *What happens with a batch where every prompt has the same length?*
  All 8 answers are good.

# Ticket 2: the first token is always right

**Severity:** medium. **Reported by:** the evaluation team.

> After the fix for the right padding, the batched answers look fluent.
> But the benchmark score in batch mode is 4 points lower than one at a
> time.

**Evidence**

- The code pads on the left now. The prefill passes the mask. The decode
  loop:

  ```python
  out = model(batch.input_ids, attention_mask=batch.attention_mask, use_cache=True)
  cache = out.past_key_values
  next_tokens = out.logits[:, -1:].argmax(-1)
  for _ in range(max_tokens - 1):
      out = model(next_tokens, past_key_values=cache, use_cache=True)
      cache = out.past_key_values
      next_tokens = out.logits[:, -1:].argmax(-1)
  ```

- The team compared each row of a batch with the same prompt alone, in
  fp32:

  | row | pads | first different token |
  |---|---|---|
  | 0 | 0 | none |
  | 1 | 3 | 2 |
  | 2 | 11 | 1 |
  | 3 | 0 | none |
  | 4 | 19 | 1 |

- The team thinks that this is the bf16 batch effect of Part 1.

### Solution

- **Root cause.** The prefill passes the mask, and the decode steps do
  not. From the first decode step on, the new token attends to every
  position of the cache, and that includes the pad positions of its row.
- **The number.** Token 0 comes from the prefill, and it is correct in
  every row. The differences start at token 1 or 2, and only in the rows
  with pads. The rows with 0 pads match exactly in fp32. The bf16 batch
  effect of Part 1 cannot explain a match that is exact, in fp32, and
  depends on the number of pads.
- **The fix.** Keep the mask, and add one column of ones for each new
  token:
  `mask = torch.cat([mask, mask.new_ones(B, 1)], dim=1)`. Pass it at
  every step.
- **The guard.** Assert that `mask.shape[1]` equals the cache length
  plus the new tokens. Compare each row with the row alone, in fp32, for
  20 tokens, not only for the first one.

**The noise.** The theory of the bf16 batch effect. It is a real effect,
but this test runs in fp32, and in fp32 that effect does not exist.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *Is the first generated token correct for every row?*
  Yes. In every row, token 0 matches the row run alone.
- *What shape does the attention mask have in the decode steps?*
  The decode steps do not pass a mask.
- *Is the comparison done in fp32 or in bf16?*
  In fp32. In bf16 the rows with 0 pads also match.

# Ticket 3: batch 256 is only 19% faster than batch 128

**Severity:** low. **Reported by:** the performance team.

> From batch 1 to batch 16, the throughput grew almost 16x. From batch
> 128 to batch 256 it grew only 1.19x. The scheduler must add overhead
> at a large batch.

**Evidence**

- Qwen3-1.7B on an L40S. Each sequence has about 512 tokens of context
  during the test.
- The measured decode throughput:

  | batch | tokens/s |
  |---|---|
  | 64 | 6,100 |
  | 128 | 8,050 |
  | 256 | 9,560 |

- The profiler: 92% of each step at batch 256 is GPU kernels.
- A colleague says: "At batch 256 the card is compute-bound. That is the
  roofline."

### Solution: nothing is broken

- **Root cause.** A batch shares the read of the **weights**. It does
  not share the read of the **KV cache**, because each sequence has its
  own. At batch 256 with 512 tokens of context, the KV cache is 15.0 GB
  for each step, and the weights are only 3.44 GB. Twice the batch reads
  almost twice the bytes.
- **The number.** The floor at 80% of the bandwidth predicts 6,146,
  8,075 and 9,579 tokens/s. The measurement is within 1% of each value.
  The predicted gain from 128 to 256 is 1.19x, and the measured gain is
  1.19x.
- **The fix.** No fix in the scheduler. To go further, read fewer KV
  bytes: an FP8 KV cache (Part 7), or a model with fewer KV bytes for
  each token.
- **The guard.** Before you tune, compute the floor with both terms. If
  the measurement is near the floor, the only way forward is fewer
  bytes.

**The noise.** The compute theory. At batch 256 a step needs
2 x 1.72 G x 256 = 0.88 TFLOP, which is 2.4 ms at 362 TFLOP/s. The step
takes 26.8 ms. Compute is far from the limit.

In [ ]:
weights, kv_per_token, context, bandwidth = 3.44e9, 114_688, 512, 864e9
for batch, measured in [(64, 6100), (128, 8050), (256, 9560)]:
    step = (weights + batch * context * kv_per_token) / (0.8 * bandwidth)
    compute = 2 * 1.72e9 * batch / 362e12
    print(f'batch {batch:3d}: KV {batch * context * kv_per_token / 1e9:5.2f} GB, '
          f'predicted {batch / step:6.0f} tok/s, measured {measured}, compute alone {compute * 1e3:.1f} ms')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How long is one step at batch 256?*
  26.8 ms.
- *How much time do the attention kernels take in that step?*
  About 80% of the kernel time.
- *How much time does the scheduler take in each step?*
  0.4 ms at batch 64 and 0.7 ms at batch 256.

# Ticket 4: the job that took nine times longer

**Severity:** medium. **Reported by:** the data team.

> We summarize 10,000 support tickets each night with a static batch of
> 32. The plan said 3.5 hours. It took 31 hours. The GPU was at 100%
> utilization the whole time.

**Evidence**

- The plan: each batch runs for the mean output length, 176 tokens.
- The output lengths: 96% of the summaries have about 100 tokens. 4% are
  long tickets, and their summaries run to the limit of 2,000 tokens.
- Without the long ones, the longest summary in a batch of 32 is about
  300 tokens.
- The code is the static batch of stage 04: a batch ends when its last
  sequence ends.

### Solution

- **Root cause.** A static batch runs until its slowest sequence ends.
  With 32 sequences and a 4% chance of a long one, most batches contain
  at least one long summary. Then 31 slots wait for about 1,700 steps
  and compute padding.
- **The number.** The chance that a batch has no long summary is
  0.96^32 = 27%. So a batch runs, on average,
  0.73 x 2,000 + 0.27 x 300 = 1,540 steps, not 176. The ratio is 8.7x,
  and 3.5 h x 8.7 = 30.6 h. The useful fraction of the slots is
  176 / 1,540 = 11%.
- **The fix.** Continuous batching (stage 05): a finished sequence
  leaves at once, and a waiting one takes its slot. A cheap partial fix
  for an offline job: sort the tickets by length, so that the long ones
  share batches.
- **The guard.** Measure the fraction of decoded slots that are padding,
  as stage 04 does. Plan with that fraction, not with the mean length.

**The noise.** "100% utilization". The GPU computes the padding at full
speed. Busy is not the same as useful.

In [ ]:
p_none = 0.96 ** 32
steps = (1 - p_none) * 2000 + p_none * 300
mean = 0.96 * 100 + 0.04 * 2000
print(f'P(no long summary in a batch) = {p_none:.2f}')
print(f'steps per batch {steps:.0f}, plan {mean:.0f}, ratio {steps / mean:.1f}x, '
      f'3.5 h -> {3.5 * steps / mean:.1f} h, useful slots {mean / steps:.0%}')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How many steps does a typical batch run?*
  73% of the batches run 2,000 steps. The other batches run 200 to 400
  steps.
- *How many sequences are still active in the last 1,700 steps of a long batch?*
  Usually one. Sometimes two.
- *Is the GPU slower than in the benchmark?*
  No. Each step takes the same time as in the benchmark.

# Ticket 5: continuous batching that barely helps

**Severity:** medium. **Reported by:** the platform team.

> We replaced the static batch with our continuous batching scheduler.
> The paper promises 2x to 4x. We get 1.3x. The waiting queue is always
> long, so we need more GPUs.

**Evidence**

- The scheduler log shows `running=32` at every step of the peak hour.
- The end of the step:

  ```python
  for seq in self.running:
      seq.output.append(next_tokens[seq.slot])
  self.running = [s for s in self.running
                  if len(s.output) < s.max_tokens]
  ```

- The answers that the users receive are correct. The [detokenizer](../../GLOSSARY.md#detokenization) stops
  the text at the first `<|im_end|>`.
- `max_tokens` is 1,024. The [median](../../GLOSSARY.md#percentile) answer has 180 tokens.

### Solution

- **Root cause.** The filter removes a sequence only when it reaches
  `max_tokens`. A sequence that emits the stop token stays in its slot,
  and it decodes until 1,024 tokens. The detokenizer hides the extra
  tokens, so the answers look correct.
- **The number.** 29 of the 32 running sequences have already finished.
  Only 3 of 32 slots do useful work. A median request uses 180 of its
  1,024 decode steps: 82% of the work is thrown away.
- **The fix.** Remove a sequence when it emits a stop token **or** when
  it reaches `max_tokens`. Your stage 05 checks both.
- **The guard.** Count the tokens that are decoded after a stop token.
  The count must be zero. Assert that no running sequence ends with a
  stop token at the start of a step.

**The noise.** The long queue. It is a result of the bug, not a lack of
GPUs: the slots are full of finished sequences.

In [ ]:
max_tokens, median = 1024, 180
print(f'useful slots: {3 / 32:.0%}')
print(f'work thrown away by a median request: {1 - median / max_tokens:.0%}')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What is the last token of each running sequence, at one step of the peak?*
  For 29 of the 32 running sequences, it is `<|im_end|>` (151645).
- *How many tokens does each sequence decode before it leaves?*
  1,024, every time.
- *How long does a request stay in the running set?*
  About 1,024 steps, even when its answer ended after 180 tokens.

# Ticket 6: the afternoon that gets slower and slower

**Severity:** high. **Reported by:** users.

> From 13:00 the first token takes longer and longer. At 13:30 it takes
> 6 minutes. At 14:00 it takes 12 minutes. A restart fixes it for a
> short time.

**Evidence**

- The server finishes 10 requests each second when it is full. Stage 05
  measured this.
- The arrival rate from 13:00 to 15:00 is 12 requests each second. The
  rest of the day it is 6.
- The time for each output token stays at 25 ms all afternoon.
- The KV cache is 96% full from 13:00.

### Solution: the engine is healthy, the capacity is not

- **Root cause.** From 13:00, 12 requests arrive each second and the
  server finishes only 10. The queue grows by 2 requests each second,
  and every new request waits behind the queue.
- **The number.** After t seconds the queue holds 2t requests, and the
  wait is 2t / 10 = 0.2t seconds. At 13:30, t = 1,800 s, and the wait is
  360 s = 6 minutes. At 14:00 it is 12 minutes. Both match. The time for
  each token is flat, so the engine itself is fine.
- **The fix.** More capacity for the peak: one more server gives 20
  requests each second. And admission control: when the queue predicts a
  wait above the limit, return HTTP 429 with a `Retry-After` header. A
  fast error is better than a wait of 12 minutes.
- **The guard.** Alert when the arrival rate is above the measured
  service rate for more than one minute, or on the slope of the queue
  length.

**The noise.** The restart. It empties the queue, so it looks like a
fix. It also drops every waiting request.

**The lesson.** When the arrivals exceed the service rate, no scheduler
helps. The queue grows without a limit.

In [ ]:
arrival, service = 12, 10
for minutes in (30, 60):
    t = minutes * 60
    queue = (arrival - service) * t
    print(f'after {minutes} min: queue {queue}, wait {queue / service / 60:.0f} min')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How long is the waiting queue at 13:30 and at 14:00?*
  3,600 requests at 13:30, and 7,200 at 14:00.
- *What happens after a restart?*
  The queue is empty. The clients retry, and the queue grows again at
  the same rate.
- *What happens after 15:00?*
  The queue gets shorter by 4 requests each second, and it is empty at about 16:00.

# Ticket 7: the bot forgets who said what

**Severity:** medium. **Reported by:** customer support.

> In long conversations the bot mixes up the turns. It answers questions
> that the user asked three turns ago, or it thinks that it said things
> that the user said. Single questions work.

**Evidence**

- The model is Llama-3-8B-Instruct. Its tokenizer has no pad token, so
  the code sets `tokenizer.pad_token = tokenizer.eos_token`, which is
  `<|eot_id|>` (id 128009).
- The chat template ends each turn with `<|eot_id|>`.
- The batching code builds the mask itself:

  ```python
  ids = pad_left(token_lists, pad_id=tokenizer.pad_token_id)
  mask = (ids != tokenizer.pad_token_id).long()
  ```

- The path for a single request does not build a mask.
- The team thinks that the model is weak in long conversations.

### Solution

- **Root cause.** The pad id and the end-of-turn id are the same number.
  The mask hides every position with that id, so it hides the pads
  **and** the real `<|eot_id|>` of every turn. The model cannot see where
  the turns end.
- **The number.** The row of a conversation with 5 turns has `pads + 5`
  zeros in the mask, not `pads`. Each extra zero is a turn boundary. A
  single question loses only one real token, so it mostly works.
- **The fix.** Take the mask from the tokenizer (`tokenizer(...,
  padding=True).attention_mask`), which knows which positions it added.
  Or pad with an id that never appears in a prompt.
- **The guard.** Assert that `mask.sum(1)` equals the true length of each
  row.

**The noise.** "The model is weak in long conversations". The same
conversation sent alone gets a correct answer.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *For a conversation of 5 turns, how many zeros does its row of the mask have?*
  The number of pads plus 5.
- *Does the bug happen for a conversation of 5 turns sent alone?*
  No. Alone, the answer is correct.
- *Does it happen for a single question in a batch?*
  Rarely. A single question has one `<|eot_id|>`, at the end of the
  user turn.

# Ticket 8: the fastest batch size makes users unhappy

**Severity:** medium. **Reported by:** the product team.

> We set the maximum batch to 192 because it gave the most tokens per
> second. Now users say that the text appears too slowly. The product
> promise is at least 20 tokens each second for each user.

**Evidence**

- Llama-3-8B on an A100. The measured decode steps:

  | batch | ms for each step |
  |---|---|
  | 64 | 22 |
  | 128 | 38 |
  | 192 | 55 |
  | 256 | 71 |

- The GPU utilization is 100% at 192. The team says that this is
  efficient.

### Solution: the wrong target

- **Root cause.** The team maximized the total tokens/s. The product
  promise is per user. In decode, each user gets one token for each step,
  so the speed of one user is 1 / step time. At 192 the step takes 55 ms,
  and each user gets 18 tokens/s.
- **The number.** The largest batch that keeps the promise is 128: 38 ms,
  26 tokens/s for each user, 3,368 tokens/s in total. Batch 192 gives
  3,491 tokens/s. The promise costs only 3.5% of the throughput.
- **The fix.** Set the maximum batch to about 128 (or between 128 and
  192, where the step is 50 ms). Serve more users with more replicas.
- **The guard.** Alert on the time for each output token (TPOT), p50 and
  p99, against 50 ms. Report the goodput: the throughput of the requests
  that kept the promise (Part 6).

**The noise.** 100% utilization. It tells you that the GPU is busy. It
does not tell you whether a user waits.

In [ ]:
for batch, ms in [(64, 22), (128, 38), (192, 55), (256, 71)]:
    per_user = 1000 / ms
    print(f'batch {batch:3d}: {per_user:4.1f} tok/s per user, {batch * per_user:5.0f} tok/s total',
          '' if per_user >= 20 else '  <- breaks the promise')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What is the total throughput at each batch size?*
  Nobody computed it. You have the numbers.
- *How many users are active at the peak?*
  About 180.
- *How many tokens does a user read each second?*
  Most people read 5 to 8 words each second, about 7 to 11 tokens.

### The pattern in the tickets

| The shape of the number | What it usually means | Tickets |
|---|---|---|
| The bad rows are exactly the rows with padding | Padding or the mask | 1, 2, 7 |
| The first token is right, the next ones are not | Prefill and decode do different things | 2 |
| The measurement is on the floor with both terms | Nothing is broken: physics | 3 |
| The useful fraction of the slots | Padding, or finished sequences that stay | 4, 5 |
| A queue that grows at a constant rate | Arrivals above the service rate | 6 |
| A count that is larger by exactly the number of turns | One id with two meanings | 7 |
| A good total and a bad value for each user | The wrong target | 8 |